In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    candidate_str = str(candidate)
    if candidate.exists() and candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)


In [ ]:
from pathlib import Path
import csv

from src.drive_service.logging_utils import setup_logging
from src.pipeline_paths import build_pipelines_paths
from src.pair_employee_events_from_days_raw import pair_employee_events


In [ ]:
root = "1FUosjKncLt18JzojmX8tKQm1nbgPI133"
paths = build_pipelines_paths(root)

events_name = "*.events_from_days_raw.cleaned.csv"
cleaned_events_files = sorted(Path(paths.events_output).rglob(events_name))
if not cleaned_events_files:
    raise FileNotFoundError(
        f"No cleaned events files found in {paths.events_output} with pattern {events_name}"
    )

paths.events_output, paths.shifts_output, len(cleaned_events_files), cleaned_events_files[:5]


In [ ]:
verbose = True
report_json = paths.shifts_output / "pair_employee_events_from_days_raw.report.json"

max_gap_hours = 16.0
keep_inferred_column = False

setup_logging(verbose)

report = pair_employee_events(
    input_dir=str(paths.events_output),
    output_dir=str(paths.shifts_output),
    events_name=events_name,
    report_json=str(report_json),
    max_gap_hours=max_gap_hours,
    keep_inferred_column=keep_inferred_column,
)

report["stats"]


In [ ]:
pairs_files = sorted(Path(paths.shifts_output).glob("*.pairs.csv"))
len(pairs_files), pairs_files[:5]


In [ ]:
if pairs_files:
    sample_pairs = pairs_files[0]
    with open(sample_pairs, "r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        sample_rows = []
        for i, row in enumerate(reader):
            sample_rows.append(row)
            if i >= 10:
                break
    sample_pairs, sample_rows
else:
    "No pairs CSV files generated"


In [ ]:
by_employee = report.get("by_employee", [])
missing_event_files = report.get("missing_event_files", [])
error_event_files = report.get("error_event_files", [])

{
    "report_json": str(report_json),
    "by_employee_count": len(by_employee),
    "missing_event_files_count": len(missing_event_files),
    "error_event_files_count": len(error_event_files),
    "by_employee_preview": by_employee[:3],
    "missing_event_files_preview": missing_event_files[:3],
    "error_event_files_preview": error_event_files[:3],
}
